Define Eval Dataset

In [1]:
import duckdb
import anthropic
import json
import os
import re
import pandas as pd
from datetime import datetime

In [ ]:
from dotenv import load_dotenv

load_dotenv()

client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
con = duckdb.connect('/Users/isabella/data/fraud_analytics.db')

In [3]:
with open('./schema_docs/schema.md', 'r') as f:
    schema_context = f.read()

In [4]:
# Redefine run_query and review_sql in 03_eval.ipynb

def run_query(sql: str) -> str:
    try:
        result = con.execute(sql).df()
        if result.empty:
            return "Query returned no results."
        return result.to_string(index=False)
    except Exception as e:
        return f"SQL ERROR: {str(e)}"


def review_sql(sql: str, user_question: str) -> dict:
    reviewer_prompt = f"""You are an adversarial SQL reviewer for a banking fraud analytics database.
Your job is to find problems in SQL queries before they return wrong answers.

{schema_context}

Review this SQL query for the following question:
QUESTION: {user_question}
SQL:
```sql
{sql}
```

Check for these issues:
1. Wrong join type — should use INNER JOIN when fraud_labels is involved
2. Missing amount filter — spend analysis should have amount > 0
3. Aggregation errors — AVG(is_fraud) for fraud rate, not SUM
4. Wrong table used
5. NULL handling issues
6. Ambiguous or missing columns

Respond in this exact JSON format:
{{
  "approved": true or false,
  "issues": ["issue 1", "issue 2"],
  "fixed_sql": "corrected SQL here, or same SQL if no issues",
  "confidence": "high/medium/low"
}}"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=1024,
        messages=[{"role": "user", "content": reviewer_prompt}]
    )
    raw = response.content[0].text
    json_match = re.search(r'\{.*\}', raw, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group())
        except:
            pass
    return {"approved": True, "issues": [], "fixed_sql": sql, "confidence": "low"}

In [5]:
# An evaluation set consisting of 30 questions, covering varying levels of difficulty and business scenarios.
EVAL_QUESTIONS = [
    # ── Basic Single Table ──────────────────────────────────────────────
    {
        "id": "E01",
        "difficulty": "easy",
        "category": "basic",
        "question": "How many total transactions are in the dataset?",
        "ground_truth_sql": "SELECT COUNT(*) AS total FROM transactions",
        "expected_answer_contains": ["13305915", "13,305,915"]
    },
    {
        "id": "E02",
        "difficulty": "easy",
        "category": "basic",
        "question": "How many unique customers are there?",
        "ground_truth_sql": "SELECT COUNT(DISTINCT id) AS unique_customers FROM users",
        "expected_answer_contains": ["2000", "2,000"]
    },
    {
        "id": "E03",
        "difficulty": "easy",
        "category": "basic",
        "question": "What is the overall fraud rate as a percentage?",
        "ground_truth_sql": "SELECT ROUND(AVG(is_fraud)*100, 4) AS fraud_rate_pct FROM fraud_labels",
        "expected_answer_contains": ["0.14", "0.15"]
    },
    {
        "id": "E04",
        "difficulty": "easy",
        "category": "basic",
        "question": "What are the different card brands available?",
        "ground_truth_sql": "SELECT DISTINCT card_brand FROM cards ORDER BY card_brand",
        "expected_answer_contains": ["Visa", "Mastercard", "Amex", "Discover"]
    },
    {
        "id": "E05",
        "difficulty": "easy",
        "category": "basic",
        "question": "What payment methods are used in transactions?",
        "ground_truth_sql": "SELECT DISTINCT use_chip FROM transactions",
        "expected_answer_contains": ["Swipe", "Chip", "Online"]
    },

    # ── Aggregate Analysis ──────────────────────────────────────────────
    {
        "id": "E06",
        "difficulty": "medium",
        "category": "aggregation",
        "question": "What is the average credit score of customers?",
        "ground_truth_sql": "SELECT ROUND(AVG(credit_score), 1) AS avg_credit_score FROM users",
        "expected_answer_contains": ["6", "7"]  # 600-799 range
    },
    {
        "id": "E07",
        "difficulty": "medium",
        "category": "aggregation",
        "question": "What is the total number of fraudulent transactions?",
        "ground_truth_sql": "SELECT SUM(is_fraud) AS total_fraud FROM fraud_labels",
        "expected_answer_contains": ["13332", "13,332"]
    },
    {
        "id": "E08",
        "difficulty": "medium",
        "category": "aggregation",
        "question": "Which year had the most transactions?",
        "ground_truth_sql": """
            SELECT YEAR(CAST(date AS TIMESTAMP)) AS year, COUNT(*) AS txn_count
            FROM transactions
            GROUP BY year ORDER BY txn_count DESC LIMIT 1
        """,
        "expected_answer_contains": ["201"]  # 2010s
    },
    {
        "id": "E09",
        "difficulty": "medium",
        "category": "aggregation",
        "question": "What percentage of cards have been found on the dark web?",
        "ground_truth_sql": """
            SELECT ROUND(SUM(CASE WHEN card_on_dark_web='Yes' THEN 1 ELSE 0 END)*100.0/COUNT(*), 2) AS pct
            FROM cards
        """,
        "expected_answer_contains": ["%", "percent", "pct", "."]
    },
    {
        "id": "E10",
        "difficulty": "medium",
        "category": "aggregation",
        "question": "What is the average number of credit cards per customer?",
        "ground_truth_sql": "SELECT ROUND(AVG(num_credit_cards), 2) FROM users",
        "expected_answer_contains": ["2", "3", "4"]
    },

    # ── Multiple Tables JOIN ──────────────────────────────────────────────
    {
        "id": "E11",
        "difficulty": "medium",
        "category": "join",
        "question": "What is the fraud rate by card type (Credit vs Debit)?",
        "ground_truth_sql": """
            SELECT c.card_type, ROUND(AVG(fl.is_fraud)*100, 3) AS fraud_rate_pct
            FROM transactions t
            JOIN cards c ON t.card_id = c.id
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY c.card_type ORDER BY fraud_rate_pct DESC
        """,
        "expected_answer_contains": ["Credit", "Debit"]
    },
    {
        "id": "E12",
        "difficulty": "medium",
        "category": "join",
        "question": "Which merchant category has the highest fraud rate?",
        "ground_truth_sql": """
            SELECT m.description, ROUND(AVG(fl.is_fraud)*100, 3) AS fraud_rate_pct
            FROM transactions t
            JOIN mcc_codes m ON t.mcc = m.mcc
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY m.description ORDER BY fraud_rate_pct DESC LIMIT 1
        """,
        "expected_answer_contains": ["%", "percent", "fraud"]
    },
    {
        "id": "E13",
        "difficulty": "medium",
        "category": "join",
        "question": "What is the average transaction amount for male vs female customers?",
        "ground_truth_sql": """
            SELECT u.gender, ROUND(AVG(t.amount), 2) AS avg_amount
            FROM transactions t
            JOIN users u ON t.client_id = u.id
            WHERE t.amount > 0
            GROUP BY u.gender
        """,
        "expected_answer_contains": ["Male", "Female"]
    },
    {
        "id": "E14",
        "difficulty": "medium",
        "category": "join",
        "question": "How many transactions were made using chip vs swipe vs online?",
        "ground_truth_sql": """
            SELECT use_chip, COUNT(*) AS txn_count
            FROM transactions
            GROUP BY use_chip ORDER BY txn_count DESC
        """,
        "expected_answer_contains": ["Swipe", "Chip", "Online"]
    },
    {
        "id": "E15",
        "difficulty": "medium",
        "category": "join",
        "question": "What is the fraud rate for online transactions vs chip vs swipe?",
        "ground_truth_sql": """
            SELECT t.use_chip, ROUND(AVG(fl.is_fraud)*100, 4) AS fraud_rate_pct
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY t.use_chip ORDER BY fraud_rate_pct DESC
        """,
        "expected_answer_contains": ["Online", "Swipe", "Chip"]
    },

    # ── Fraud Analysis ──────────────────────────────────────────────
    {
        "id": "E16",
        "difficulty": "hard",
        "category": "fraud",
        "question": "What are the top 3 merchant categories by total fraud amount?",
        "ground_truth_sql": """
            SELECT m.description,
                   SUM(CASE WHEN fl.is_fraud=1 THEN t.amount ELSE 0 END) AS total_fraud_amount
            FROM transactions t
            JOIN mcc_codes m ON t.mcc = m.mcc
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            WHERE t.amount > 0
            GROUP BY m.description ORDER BY total_fraud_amount DESC LIMIT 3
        """,
        "expected_answer_contains": ["$", "fraud"]
    },
    {
        "id": "E17",
        "difficulty": "hard",
        "category": "fraud",
        "question": "Do customers with higher credit scores have lower fraud rates?",
        "ground_truth_sql": """
            SELECT
                CASE
                    WHEN u.credit_score >= 750 THEN 'Excellent (750+)'
                    WHEN u.credit_score >= 670 THEN 'Good (670-749)'
                    WHEN u.credit_score >= 580 THEN 'Fair (580-669)'
                    ELSE 'Poor (<580)'
                END AS credit_tier,
                ROUND(AVG(fl.is_fraud)*100, 4) AS fraud_rate_pct
            FROM transactions t
            JOIN users u ON t.client_id = u.id
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY credit_tier ORDER BY fraud_rate_pct DESC
        """,
        "expected_answer_contains": ["credit", "fraud"]
    },
    {
        "id": "E18",
        "difficulty": "hard",
        "category": "fraud",
        "question": "What is the average fraudulent transaction amount vs legitimate transaction amount?",
        "ground_truth_sql": """
            SELECT fl.is_fraud,
                   ROUND(AVG(t.amount), 2) AS avg_amount,
                   COUNT(*) AS txn_count
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            WHERE t.amount > 0
            GROUP BY fl.is_fraud
        """,
        "expected_answer_contains": ["fraud", "legitimate", "$"]
    },
    {
        "id": "E19",
        "difficulty": "hard",
        "category": "fraud",
        "question": "Which hour of the day has the highest fraud rate?",
        "ground_truth_sql": """
            SELECT HOUR(CAST(date AS TIMESTAMP)) AS hour,
                   ROUND(AVG(fl.is_fraud)*100, 4) AS fraud_rate_pct,
                   COUNT(*) AS total_txns
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY hour ORDER BY fraud_rate_pct DESC LIMIT 5
        """,
        "expected_answer_contains": ["hour", "AM", "PM", ":00"]
    },
    {
        "id": "E20",
        "difficulty": "hard",
        "category": "fraud",
        "question": "What percentage of fraud occurs on transactions with errors?",
        "ground_truth_sql": """
            SELECT
                CASE WHEN t.errors IS NULL THEN 'No Error' ELSE 'Has Error' END AS error_flag,
                COUNT(*) AS total,
                ROUND(AVG(fl.is_fraud)*100, 3) AS fraud_rate_pct
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY error_flag
        """,
        "expected_answer_contains": ["error", "Error", "%"]
    },

    # ── Customer Persona ──────────────────────────────────────────────
    {
        "id": "E21",
        "difficulty": "hard",
        "category": "customer",
        "question": "What is the gender breakdown of customers?",
        "ground_truth_sql": """
            SELECT gender, COUNT(*) AS count,
                   ROUND(COUNT(*)*100.0/(SELECT COUNT(*) FROM users), 1) AS pct
            FROM users GROUP BY gender
        """,
        "expected_answer_contains": ["Male", "Female"]
    },
    {
        "id": "E22",
        "difficulty": "hard",
        "category": "customer",
        "question": "What is the average yearly income of customers who experienced fraud vs those who didn't?",
        "ground_truth_sql": """
            SELECT
                MAX(fl.is_fraud) AS had_fraud,
                ROUND(AVG(u.yearly_income), 0) AS avg_income
            FROM users u
            JOIN transactions t ON u.id = t.client_id
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY u.id
            ORDER BY had_fraud DESC
        """,
        "expected_answer_contains": ["income", "$", "fraud"]
    },
    {
        "id": "E23",
        "difficulty": "hard",
        "category": "customer",
        "question": "Which age group has the most transactions?",
        "ground_truth_sql": """
            SELECT
                CASE
                    WHEN u.current_age < 30 THEN 'Under 30'
                    WHEN u.current_age < 45 THEN '30-44'
                    WHEN u.current_age < 60 THEN '45-59'
                    ELSE '60+'
                END AS age_group,
                COUNT(*) AS txn_count
            FROM transactions t
            JOIN users u ON t.client_id = u.id
            GROUP BY age_group ORDER BY txn_count DESC
        """,
        "expected_answer_contains": ["30", "45", "60"]
    },
    {
        "id": "E24",
        "difficulty": "hard",
        "category": "customer",
        "question": "What is the total debt to income ratio on average across all customers?",
        "ground_truth_sql": """
            SELECT ROUND(AVG(total_debt / NULLIF(yearly_income, 0)), 3) AS avg_dti_ratio
            FROM users
        """,
        "expected_answer_contains": ["."]
    },
    {
        "id": "E25",
        "difficulty": "hard",
        "category": "customer",
        "question": "How many customers have more than 3 credit cards?",
        "ground_truth_sql": """
            SELECT COUNT(*) AS customers_with_4plus_cards
            FROM users WHERE num_credit_cards > 3
        """,
        "expected_answer_contains": ["customer", "card"]
    },

    # ── Trend Analysis ──────────────────────────────────────────────
    {
        "id": "E26",
        "difficulty": "hard",
        "category": "trend",
        "question": "How has the total transaction volume changed year over year?",
        "ground_truth_sql": """
            SELECT YEAR(CAST(date AS TIMESTAMP)) AS year,
                   COUNT(*) AS txn_count,
                   ROUND(SUM(CASE WHEN amount > 0 THEN amount ELSE 0 END), 0) AS total_volume
            FROM transactions
            GROUP BY year ORDER BY year
        """,
        "expected_answer_contains": ["2010", "201"]
    },
    {
        "id": "E27",
        "difficulty": "hard",
        "category": "trend",
        "question": "Which month of the year has the highest average transaction amount?",
        "ground_truth_sql": """
            SELECT MONTH(CAST(date AS TIMESTAMP)) AS month,
                   ROUND(AVG(amount), 2) AS avg_amount
            FROM transactions WHERE amount > 0
            GROUP BY month ORDER BY avg_amount DESC LIMIT 1
        """,
        "expected_answer_contains": ["January", "February", "March", "April", "May",
                                      "June", "July", "August", "September",
                                      "October", "November", "December", "month"]
    },
    {
        "id": "E28",
        "difficulty": "hard",
        "category": "trend",
        "question": "What is the fraud rate trend by year?",
        "ground_truth_sql": """
            SELECT YEAR(CAST(t.date AS TIMESTAMP)) AS year,
                   ROUND(AVG(fl.is_fraud)*100, 4) AS fraud_rate_pct
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY year ORDER BY year
        """,
        "expected_answer_contains": ["2010", "fraud", "%"]
    },
    {
        "id": "E29",
        "difficulty": "hard",
        "category": "trend",
        "question": "What is the most common transaction error type?",
        "ground_truth_sql": """
            SELECT errors, COUNT(*) AS count
            FROM transactions
            WHERE errors IS NOT NULL
            GROUP BY errors ORDER BY count DESC LIMIT 5
        """,
        "expected_answer_contains": ["error", "Error", "insufficient", "declined"]
    },
    {
        "id": "E30",
        "difficulty": "hard",
        "category": "trend",
        "question": "How does fraud rate differ between transactions with errors and those without?",
        "ground_truth_sql": """
            SELECT
                CASE WHEN t.errors IS NULL THEN 'No Error' ELSE 'Has Error' END AS has_error,
                COUNT(*) AS total_txns,
                ROUND(AVG(fl.is_fraud)*100, 3) AS fraud_rate_pct
            FROM transactions t
            JOIN fraud_labels fl ON t.id = fl.transaction_id
            GROUP BY has_error ORDER BY fraud_rate_pct DESC
        """,
        "expected_answer_contains": ["error", "fraud", "%"]
    },
]

print(f"✓ Eval set loaded: {len(EVAL_QUESTIONS)} questions")
print(f"  Easy:   {sum(1 for q in EVAL_QUESTIONS if q['difficulty']=='easy')}")
print(f"  Medium: {sum(1 for q in EVAL_QUESTIONS if q['difficulty']=='medium')}")
print(f"  Hard:   {sum(1 for q in EVAL_QUESTIONS if q['difficulty']=='hard')}")

✓ Eval set loaded: 30 questions
  Easy:   5
  Medium: 10
  Hard:   15


Run Eval

In [6]:
def evaluate_agent(questions: list, use_reviewer: bool = True) -> pd.DataFrame:
    """Run the whole eval，return result DataFrame"""
    
    system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
Given a question, write SQL to answer it, then summarize the result.

{schema_context}

Rules:
- Always wrap SQL in ```sql ... ``` blocks
- amount can be negative (refunds) — filter amount > 0 for spend analysis
- For fraud rate: use AVG(is_fraud)*100
- Always INNER JOIN fraud_labels (never LEFT JOIN)
"""
    
    results = []
    
    for i, q in enumerate(questions):
        print(f"[{i+1}/{len(questions)}] {q['id']} — {q['question'][:60]}...")
        
        try:
            # Step 1: Generate SQL
            response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=1024,
                system=system_prompt,
                messages=[{"role": "user", "content": q['question']}]
            )
            agent_reply = response.content[0].text
            sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
            
            if not sql_match:
                results.append({**q, "status": "NO_SQL", "passed": False,
                                 "agent_answer": agent_reply, "sql_used": ""})
                continue
            
            sql = sql_match.group(1).strip()
            reviewed = False
            issues_found = []
            
            # Step 2: Review（Optional）
            if use_reviewer:
                review = review_sql(sql, q['question'])
                if not review['approved']:
                    sql = review['fixed_sql']
                    reviewed = True
                    issues_found = review['issues']
            
            # Step 3: Execute SQL
            query_result = run_query(sql)
            if query_result.startswith("SQL ERROR"):
                results.append({**q, "status": "SQL_ERROR", "passed": False,
                                 "agent_answer": query_result, "sql_used": sql,
                                 "reviewed": reviewed, "issues": issues_found})
                continue
            
            # Step 4: Generate insight
            interp = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=256,
                messages=[{
                    "role": "user",
                    "content": f"Question: {q['question']}\nResult:\n{query_result}\nSummarize in 1-2 sentences."
                }]
            )
            agent_answer = interp.content[0].text
            
            # Step 5: Test if pass
            answer_lower = agent_answer.lower() + query_result.lower()
            passed = any(
                expected.lower() in answer_lower
                for expected in q['expected_answer_contains']
            )
            
            results.append({
                "id": q['id'],
                "difficulty": q['difficulty'],
                "category": q['category'],
                "question": q['question'],
                "passed": passed,
                "status": "PASS" if passed else "FAIL",
                "reviewed": reviewed,
                "issues": issues_found,
                "sql_used": sql,
                "agent_answer": agent_answer[:200],
            })
            
        except Exception as e:
            results.append({**q, "status": "ERROR", "passed": False,
                             "agent_answer": str(e), "sql_used": ""})
    
    return pd.DataFrame(results)


# Run eval（30 questions may take 3-5 minutes）
print("Starting eval run...\n")
results_df = evaluate_agent(EVAL_QUESTIONS, use_reviewer=True)

Starting eval run...

[1/30] E01 — How many total transactions are in the dataset?...
[2/30] E02 — How many unique customers are there?...
[3/30] E03 — What is the overall fraud rate as a percentage?...
[4/30] E04 — What are the different card brands available?...
[5/30] E05 — What payment methods are used in transactions?...
[6/30] E06 — What is the average credit score of customers?...
[7/30] E07 — What is the total number of fraudulent transactions?...
[8/30] E08 — Which year had the most transactions?...
[9/30] E09 — What percentage of cards have been found on the dark web?...
[10/30] E10 — What is the average number of credit cards per customer?...
[11/30] E11 — What is the fraud rate by card type (Credit vs Debit)?...
[12/30] E12 — Which merchant category has the highest fraud rate?...
[13/30] E13 — What is the average transaction amount for male vs female cu...
[14/30] E14 — How many transactions were made using chip vs swipe vs onlin...
[15/30] E15 — What is the fraud rate for 

Print Result Report

In [7]:
def print_eval_report(df: pd.DataFrame):
    total = len(df)
    passed = df['passed'].sum()
    accuracy = passed / total * 100
    
    print("=" * 55)
    print(f"  EVAL REPORT — {datetime.now().strftime('%Y-%m-%d %H:%M')}")
    print("=" * 55)
    print(f"  Overall Accuracy : {passed}/{total} = {accuracy:.1f}%")
    print()
    
    # By Difficulty
    print("  By Difficulty:")
    for diff in ['easy', 'medium', 'hard']:
        sub = df[df['difficulty'] == diff]
        p = sub['passed'].sum()
        t = len(sub)
        print(f"    {diff:8s}: {p}/{t} ({p/t*100:.0f}%)")
    
    # By Category
    print("\n  By Category:")
    for cat in df['category'].unique():
        sub = df[df['category'] == cat]
        p = sub['passed'].sum()
        t = len(sub)
        print(f"    {cat:12s}: {p}/{t} ({p/t*100:.0f}%)")
    
    # reviewer actiivty level
    reviewed_count = df['reviewed'].sum() if 'reviewed' in df.columns else 0
    print(f"\n  Reviewer fixed  : {reviewed_count} queries")
    
    # Failed Questions
    failed = df[df['passed'] == False]
    if len(failed) > 0:
        print(f"\n  Failed Questions:")
        for _, row in failed.iterrows():
            print(f"    ✗ [{row['id']}] {row['question'][:55]}...")
    
    print("=" * 55)
    
    # Save the result
    timestamp = datetime.now().strftime('%Y%m%d_%H%M')
    df.to_csv(f'./eval_results_{timestamp}.csv', index=False)
    print(f"\n  Results saved to eval_results_{timestamp}.csv")

print_eval_report(results_df)

  EVAL REPORT — 2026-06-08 09:54
  Overall Accuracy : 30/30 = 100.0%

  By Difficulty:
    easy    : 5/5 (100%)
    medium  : 10/10 (100%)
    hard    : 15/15 (100%)

  By Category:
    basic       : 5/5 (100%)
    aggregation : 5/5 (100%)
    join        : 5/5 (100%)
    fraud       : 5/5 (100%)
    customer    : 5/5 (100%)
    trend       : 5/5 (100%)

  Reviewer fixed  : 6 queries

  Results saved to eval_results_20260608_0954.csv


In [8]:
# Run eval without reviewer
print("Running eval WITHOUT reviewer...\n")
results_no_reviewer = evaluate_agent(EVAL_QUESTIONS, use_reviewer=False)
results_no_reviewer.to_csv('./eval_results_no_reviewer.csv', index=False)
print_eval_report(results_no_reviewer)

Running eval WITHOUT reviewer...

[1/30] E01 — How many total transactions are in the dataset?...
[2/30] E02 — How many unique customers are there?...
[3/30] E03 — What is the overall fraud rate as a percentage?...
[4/30] E04 — What are the different card brands available?...
[5/30] E05 — What payment methods are used in transactions?...
[6/30] E06 — What is the average credit score of customers?...
[7/30] E07 — What is the total number of fraudulent transactions?...
[8/30] E08 — Which year had the most transactions?...
[9/30] E09 — What percentage of cards have been found on the dark web?...
[10/30] E10 — What is the average number of credit cards per customer?...
[11/30] E11 — What is the fraud rate by card type (Credit vs Debit)?...
[12/30] E12 — Which merchant category has the highest fraud rate?...
[13/30] E13 — What is the average transaction amount for male vs female cu...
[14/30] E14 — How many transactions were made using chip vs swipe vs onlin...
[15/30] E15 — What is the fra

In [11]:
#Analyze what reviewered fixed
#Re-run with reviewer, documenting detailed review info
def evaluate_with_review_details(questions):
    """Run eval and document details of each reviewer decision"""
    
    system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
Given a question, write SQL to answer it.

{schema_context}

Always wrap SQL in ```sql ... ``` blocks.
- amount can be negative (refunds) — filter amount > 0 for spend analysis
- For fraud rate: use AVG(is_fraud)*100
- Always INNER JOIN fraud_labels (never LEFT JOIN)
"""
    
    records = []
    
    for i, q in enumerate(questions):
        print(f"[{i+1}/{len(questions)}] {q['id']}...")
        
        try:
            # 生成SQL
            response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=1024,
                system=system_prompt,
                messages=[{"role": "user", "content": q['question']}]
            )
            agent_reply = response.content[0].text
            sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
            if not sql_match:
                continue
            
            original_sql = sql_match.group(1).strip()
            
            # Review
            review = review_sql(original_sql, q['question'])
            
            records.append({
                "id": q['id'],
                "difficulty": q['difficulty'],
                "category": q['category'],
                "question": q['question'],
                "original_sql": original_sql,
                "approved": review['approved'],
                "issues": " | ".join(review['issues']) if review['issues'] else "",
                "fixed_sql": review['fixed_sql'],
                "confidence": review.get('confidence', ''),
                "sql_changed": original_sql.strip() != review['fixed_sql'].strip()
            })
            
        except Exception as e:
            print(f"  Error: {e}")
    
    return pd.DataFrame(records)

print("Running detailed review analysis...\n")
review_details = evaluate_with_review_details(EVAL_QUESTIONS)
review_details.to_csv('./review_details.csv', index=False)
print(f"\n✓ Done. Total queries reviewed: {len(review_details)}")
print(f"  Approved as-is: {review_details['approved'].sum()}")
print(f"  Fixed by reviewer: {(~review_details['approved']).sum()}")
print(f"\nFixed queries:")
fixed = review_details[~review_details['approved']]
for _, row in fixed.iterrows():
    print(f"\n  [{row['id']}] {row['question'][:55]}...")
    print(f"  Issue: {row['issues']}")

Running detailed review analysis...

[1/30] E01...
[2/30] E02...
[3/30] E03...
[4/30] E04...
[5/30] E05...
[6/30] E06...
[7/30] E07...
[8/30] E08...
[9/30] E09...
[10/30] E10...
[11/30] E11...
[12/30] E12...
[13/30] E13...
[14/30] E14...
[15/30] E15...
[16/30] E16...
[17/30] E17...
[18/30] E18...
[19/30] E19...
[20/30] E20...
[21/30] E21...
[22/30] E22...
[23/30] E23...
[24/30] E24...
[25/30] E25...
[26/30] E26...
[27/30] E27...
[28/30] E28...
[29/30] E29...
[30/30] E30...

✓ Done. Total queries reviewed: 30
  Approved as-is: 26
  Fixed by reviewer: 4

Fixed queries:

  [E22] What is the average yearly income of customers who expe...
  Issue: Customer classification is incorrect: A customer who has both fraud and non-fraud transactions will be counted in BOTH groups (once per transaction type), inflating customer counts and biasing income averages | The AVG(u.yearly_income) is transaction-weighted, not customer-weighted: customers with more transactions have disproportionate influence 

In [12]:
def evaluate_agent_v2(questions: list, use_reviewer: bool = True) -> pd.DataFrame:
    """
    Pro eval：Record review details at the same time，self-consistent data
    """
    
    system_prompt = f"""You are a data analyst for a banking fraud analytics platform.
Given a question, write SQL to answer it.

{schema_context}

Rules:
- Always wrap SQL in ```sql ... ``` blocks
- amount can be negative (refunds) — filter amount > 0 for spend analysis
- For fraud rate: use AVG(is_fraud)*100
- Always INNER JOIN fraud_labels (never LEFT JOIN)
"""
    
    results = []
    
    for i, q in enumerate(questions):
        print(f"[{i+1}/{len(questions)}] {q['id']} — {q['question'][:55]}...")
        
        try:
            # Step 1: Generate SQL
            response = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=1024,
                system=system_prompt,
                messages=[{"role": "user", "content": q['question']}]
            )
            agent_reply = response.content[0].text
            sql_match = re.search(r'```sql\s*(.*?)\s*```', agent_reply, re.DOTALL)
            
            if not sql_match:
                results.append({
                    "id": q['id'], "difficulty": q['difficulty'],
                    "category": q['category'], "question": q['question'],
                    "passed": False, "status": "NO_SQL",
                    "reviewed": False, "issues": "",
                    "original_sql": "", "final_sql": "",
                    "agent_answer": agent_reply[:200]
                })
                continue
            
            original_sql = sql_match.group(1).strip()
            final_sql = original_sql
            reviewed = False
            issues = ""
            
            # Step 2: Review（Optional）
            if use_reviewer:
                review = review_sql(original_sql, q['question'])
                reviewed = not review['approved']
                issues = " | ".join(review['issues']) if review['issues'] else ""
                if not review['approved']:
                    final_sql = review['fixed_sql']
            
            # Step 3: Execute SQL
            query_result = run_query(final_sql)
            if query_result.startswith("SQL ERROR"):
                results.append({
                    "id": q['id'], "difficulty": q['difficulty'],
                    "category": q['category'], "question": q['question'],
                    "passed": False, "status": "SQL_ERROR",
                    "reviewed": reviewed, "issues": issues,
                    "original_sql": original_sql, "final_sql": final_sql,
                    "agent_answer": query_result[:200]
                })
                continue
            
            # Step 4: Generate insight
            interp = client.messages.create(
                model="claude-sonnet-4-5",
                max_tokens=256,
                messages=[{
                    "role": "user",
                    "content": f"Question: {q['question']}\nResult:\n{query_result}\nSummarize in 1-2 sentences."
                }]
            )
            agent_answer = interp.content[0].text
            
            # Step 5: Judge pass/fail
            answer_lower = agent_answer.lower() + query_result.lower()
            passed = any(
                exp.lower() in answer_lower
                for exp in q['expected_answer_contains']
            )
            
            results.append({
                "id": q['id'],
                "difficulty": q['difficulty'],
                "category": q['category'],
                "question": q['question'],
                "passed": passed,
                "status": "PASS" if passed else "FAIL",
                "reviewed": reviewed,
                "issues": issues,
                "original_sql": original_sql,
                "final_sql": final_sql,
                "agent_answer": agent_answer[:200]
            })
            
        except Exception as e:
            results.append({
                "id": q['id'], "difficulty": q['difficulty'],
                "category": q['category'], "question": q['question'],
                "passed": False, "status": "ERROR",
                "reviewed": False, "issues": str(e),
                "original_sql": "", "final_sql": "",
                "agent_answer": ""
            })
    
    return pd.DataFrame(results)

In [13]:
print("=" * 55)
print("RUN A: With Reviewer")
print("=" * 55)
results_with = evaluate_agent_v2(EVAL_QUESTIONS, use_reviewer=True)
results_with['run'] = 'with_reviewer'
results_with.to_csv('./final_with_reviewer.csv', index=False)
print_eval_report(results_with)

print("\n\n" + "=" * 55)
print("RUN B: Without Reviewer")
print("=" * 55)
results_without = evaluate_agent_v2(EVAL_QUESTIONS, use_reviewer=False)
results_without['run'] = 'without_reviewer'
results_without.to_csv('./final_without_reviewer.csv', index=False)
print_eval_report(results_without)

RUN A: With Reviewer
[1/30] E01 — How many total transactions are in the dataset?...
[2/30] E02 — How many unique customers are there?...
[3/30] E03 — What is the overall fraud rate as a percentage?...
[4/30] E04 — What are the different card brands available?...
[5/30] E05 — What payment methods are used in transactions?...
[6/30] E06 — What is the average credit score of customers?...
[7/30] E07 — What is the total number of fraudulent transactions?...
[8/30] E08 — Which year had the most transactions?...
[9/30] E09 — What percentage of cards have been found on the dark we...
[10/30] E10 — What is the average number of credit cards per customer...
[11/30] E11 — What is the fraud rate by card type (Credit vs Debit)?...
[12/30] E12 — Which merchant category has the highest fraud rate?...
[13/30] E13 — What is the average transaction amount for male vs fema...
[14/30] E14 — How many transactions were made using chip vs swipe vs ...
[15/30] E15 — What is the fraud rate for online transac

In [14]:
results_with = pd.read_csv('./final_with_reviewer.csv')

fixed = results_with[results_with['reviewed'] == True]
print(f"Queries fixed by reviewer: {len(fixed)}\n")
for _, row in fixed.iterrows():
    print(f"[{row['id']}] {row['question']}")
    print(f"Issue: {row['issues']}")
    print(f"\nOriginal SQL:\n{row['original_sql']}")
    print(f"\nFixed SQL:\n{row['final_sql']}")
    print("-" * 60)

Queries fixed by reviewer: 3

[E19] Which hour of the day has the highest fraud rate?
Issue: Ambiguous column reference: 'is_fraud' should be qualified as 'fl.is_fraud' since it comes from the fraud_labels table | Missing amount filter: While the question doesn't explicitly ask for 'spend analysis', fraud rate analysis typically excludes refunds (negative amounts) to avoid skewing patterns, though this is a minor issue since the question is about fraud rate by hour

Original SQL:
SELECT
    EXTRACT(HOUR FROM CAST(date AS TIMESTAMP)) AS hour_of_day,
    COUNT(*) AS total_txns,
    SUM(is_fraud) AS fraud_count,
    AVG(is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fraud_labels fl ON t.id = fl.transaction_id
GROUP BY hour_of_day
ORDER BY fraud_rate_pct DESC

Fixed SQL:
SELECT
    EXTRACT(HOUR FROM CAST(t.date AS TIMESTAMP)) AS hour_of_day,
    COUNT(*) AS total_txns,
    SUM(fl.is_fraud) AS fraud_count,
    AVG(fl.is_fraud) * 100 AS fraud_rate_pct
FROM transactions t
JOIN fra